In [1]:
import os
from collections.abc import Callable

import pandas as pd

from utils import PROCESSED_DIR, clean_styler

# Directory containing raw observations
DATA_DIR = "data/original"

# Overwrite outputs
OVERWRITE = True

# Fu2023's assumed mass (µg) per particle
PARTICLE_MASS = {"land": 0.057, "ocean": 0.1}


def annotate_fu2023_obs(path: str, metadata: dict) -> pd.DataFrame:
    """Load observed concentration or deposition and add metadata"""

    # Define column names; read concentration units from second row
    number_units, mass_units = pd.read_table(path, header=1, nrows=0).columns[-2:]
    number_units = number_units.replace("MPs/", "particle/")
    mass_units = mass_units.replace("/a", "/yr")
    cols = ["type", "year", "mon", "lon", "lat", number_units, mass_units]

    # Load data file
    obs = pd.read_table(path, header=1, names=cols)

    # Add metadata identifying source study
    for study, info in metadata.items():
        idx = info.pop("idx")
        obs.loc[idx, "study"] = study
        for col, value in info.items():
            obs.loc[idx, col] = value

    return obs


def annotate_fu2023_conc(data_dir: str) -> pd.DataFrame:
    """Load observed concentration and add metadata"""

    # Define metadata
    metadata = {
        "Allen2021": {
            "idx": 0,
            "doi": "https://doi.org/10.1038/s41467-021-27454-7",
            "size_low": 3.5,
            "size_high": 53,
            "size_units": "um",
            "shape": "Fibers + Fragments",
            "setting": "land",
            "notes": "size range from results section",
        },
        "Liu2019": {
            "idx": range(1, 4),
            "doi": "https://doi.org/10.1021/acs.est.9b03427",
            "size_low": 16,
            "size_high": 2087,
            "size_units": "um",
            "shape": "Fibers + Fragments",
            "setting": "ocean",
            "notes": "size range from results section",
        },
        "Abbasi2019": {
            "idx": 4,
            "doi": "https://doi.org/10.1016/j.envpol.2018.10.039",
            "size_low": 10,
            "size_high": 500,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range estimated from section 3.3",
        },
        "Unknown1": {"idx": 5, "setting": "land", "notes": "North of Bushehr, Iran"},
        "Allen2020": {
            "idx": 6,
            "doi": "https://doi.org/10.1371/journal.pone.0232746",
            "size_low": 5,
            "size_high": 140,
            "size_units": "um",
            "size_units": "um",
            "shape": "Not reported",
            "setting": "land",
            "notes": "size range from results section",
        },
        "Gaston2020": {
            "idx": 7,
            "doi": "https://doi.org/10.1177/0003702820920652",
            "size_low": 25,
            "size_high": 2061,
            "size_units": "um",
            "shape": "Fibers + Fragments",
            "setting": "land",
            "notes": "size range from table 2",
        },
        "Dris2017": {
            "idx": 8,
            "doi": "https://doi.org/10.1016/j.envpol.2016.12.013",
            "size_low": 50,
            "size_high": 1650,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range from figure 2",
        },
        "Unknown2": {"idx": 9, "setting": "land", "notes": "Hangzhou Bay, China"},
        "Liu2020": {
            "idx": range(10, 19),
            "doi": "https://doi.org/10.1016/j.jhazmat.2020.123223",
            "size_low": 16,
            "size_high": 2986,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range from section 3.3",
        },
        "Yuan2023": {
            "idx": 19,
            "doi": "https://doi.org/10.1016/j.scitotenv.2023.161839",
            "size_low": 13,
            "size_high": 334,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range from section 3.2",
        },
        "Ding2021": {
            "idx": 20,
            "doi": "https://doi.org/10.1016/j.atmosenv.2021.118389",
            "size_low": 50,
            "size_high": 2210,
            "size_units": "um",
            "shape": "Fibers + Films + Foams + Fragments",
            "setting": "ocean",
            "notes": "size range from section 3.2",
        },
        "Wang2021": {
            "idx": 21,
            "doi": "https://doi.org/10.1016/j.jhazmat.2021.125477",
            "size_low": 19,
            "size_high": 949,
            "size_units": "um",
            "shape": "Fibers + Films + Fragments",
            "setting": "ocean",
            "notes": "size range from section 3.2",
        },
        "Wang2020": {
            "idx": range(22, 43),
            "doi": "https://doi.org/10.1016/j.jhazmat.2019.121846",
            "size_low": 59,
            "size_high": 2252,
            "size_units": "um",
            "shape": "Fibers + Fragments",
            "setting": "ocean",
            "notes": "size range from section 3.2",
        },
        "Trainic2020": {
            "idx": range(43, 52),
            "doi": "https://doi.org/10.1038/s43247-020-00061-y",
            "size_low": 17,
            "size_high": 172,
            "size_units": "um",
            "shape": "Not reported",
            "setting": "ocean",
            "notes": "size range from table 1",
        },
        "Caracci2023": {
            "idx": range(52, 72),
            "doi": "https://doi.org/10.1016/j.jhazmat.2023.131036",
            "size_low": 1,
            "size_high": 50,
            "size_units": "um",
            "shape": "Fibers + Fragments",
            "setting": "ocean",
            "notes": (
                "size range estimated from methods (sampling with PM10 inlet head may"
                + " capture fibers longer than 10 um)"
            ),
        },
    }

    # Load and set metadata
    path = os.path.join(data_dir, "concentration.txt")
    obs = annotate_fu2023_obs(path=path, metadata=metadata)

    return obs


def annotate_fu2023_depo(data_dir: str) -> pd.DataFrame:
    """Load observed deposition and add metadata"""

    # Define metadata
    metadata = {
        # Do Liu2020 first to set dtypes for all new columns
        "Liu2020": {
            "idx": range(8, 12),
            "doi": "https://doi.org/10.1016/j.jhazmat.2020.123223",
            "size_low": 16,
            "size_high": 4900,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range estimated from figure 3b",
        },
        "Unknown1": {"idx": 0, "setting": "ocean", "notes": "Bohai Sea"},
        "Unknown2": {"idx": 1, "setting": "ocean", "notes": "Bohai Sea"},
        "Unknown3": {"idx": 2, "setting": "land", "notes": "Eastern Shandong, China"},
        "Unknown4": {"idx": 3, "setting": "land", "notes": "Tianjin, China"},
        "Unknown5": {"idx": 4, "setting": "land", "notes": "West of Dalian, China"},
        "Unknown6": {"idx": 5, "setting": "land", "notes": "Eastern Shandong, China"},
        "Unknown7": {"idx": 6, "setting": "land", "notes": "East of Dongguan, China"},
        "Unknown8": {"idx": 7, "setting": "land", "notes": "Tibet, China"},
        "Cai2017": {
            "idx": 12,
            "doi": "https://doi.org/10.1007/s11356-017-0116-x",
            "size_low": 20,
            "size_high": 5000,
            "size_units": "um",
            "shape": "Fibers + Films + Foams + Fragments",
            "setting": "land",
            "notes": "size range estimated from figure 4",
        },
        "Tian2020": {
            "idx": range(13, 16),
            "doi": "https://doi.org/10.13671/j.hjkxxb.2020.0067",
            "setting": "land",
            "notes": "could not access article",
        },
        "Brahney2020": {
            "idx": range(16, 27),
            "doi": "https://doi.org/10.1126/science.aaz5819",
            "size_low": 4,
            "size_high": 3000,
            "size_units": "um",
            "shape": "Fibers + Fragments",
            "setting": "land",
            "notes": "size range stated in paper",
        },
        "Allen2019": {
            "idx": 27,
            "doi": "https://doi.org/10.1038/s41561-019-0335-5",
            "size_low": 10,
            "size_high": 3000,
            "size_units": "um",
            "shape": "Fibers + Films + Fragments",
            "setting": "land",
            "notes": "size range estimated from figure 2",
        },
        "Klein2019": {
            "idx": range(28, 31),
            "doi": "https://doi.org/10.1016/j.scitotenv.2019.05.405",
            "size_low": 20,
            "size_high": 5000,
            "size_units": "um",
            "shape": "Fibers + Fragments",
            "setting": "land",
            "notes": "size range estimated from section 3.3",
        },
        "Roblin2020": {
            "idx": 31,
            "doi": "https://doi.org/10.1021/acs.est.0c04000",
            "size_low": 50,
            "size_high": 4000,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": (
                "size range estimated from figure S3 and statement that visual detection"
                + " was limited to fibers with length > 50 um"
            ),
        },
        "Dris2016": {
            "idx": 32,
            "doi": "https://doi.org/10.1016/j.marpolbul.2016.01.006",
            "size_low": 50,
            "size_high": 3200,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range estimated from figure 3",
        },
        "Stanton2019": {
            "idx": 33,
            "doi": "https://doi.org/10.1016/j.scitotenv.2019.02.278",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range not reported",
        },
        "Wright2020": {
            "idx": [34, 35],
            "doi": "https://doi.org/10.1016/j.envint.2019.105411",
            "size_low": 25,
            "size_high": 2900,
            "size_units": "um",
            "shape": "Fibers + Films + Foams + Fragments",
            "setting": "land",
            "notes": "size range estimated from figure 2",
        },
        "Yuan2023": {
            "idx": 36,
            "doi": "https://doi.org/10.1016/j.scitotenv.2023.161839",
            "size_low": 29,
            "size_high": 185,
            "size_units": "um",
            "shape": "Fibers",
            "setting": "land",
            "notes": "size range from section 3.2",
        },
    }

    # Load and set metadata
    path = os.path.join(data_dir, "deposition.txt")
    obs = annotate_fu2023_obs(path=path, metadata=metadata)

    return obs


def revise_fu2023_conc(obs: pd.DataFrame) -> pd.DataFrame:
    """Fix inconsistencies in original data"""

    def update(doi: str, updates: dict) -> None:
        """Update data and possibly recompute mass concentration"""

        mask = obs["doi"].fillna("Unknown").str.endswith(doi)

        if "particle/m3" in updates:
            setting = obs.loc[mask, "setting"].unique()
            if len(setting) > 1:
                raise ValueError(f"Rows to update must have same setting; got {setting}")
            setting = setting[0]

            ug_per_particle = PARTICLE_MASS[setting]
            updates["ug/m3"] = round(ug_per_particle * updates["particle/m3"], 15)

        if "notes" in updates:
            updates["notes"] = (
                obs.loc[mask, "notes"].astype(str) + "; " + updates["notes"]
            )

        for k, v in updates.items():
            obs.loc[mask, k] = v

    # Abbasi2019
    # Fix lat/lon (was mislocated); fix number (increase precision)
    update(
        doi="10.1016/j.envpol.2018.10.039",
        updates={"lat": 27.47, "lon": 52.61, "particle/m3": 0.657},
    )

    # Allen2021
    # Fix lat/lon (was mislocated); fix number (increase precision)
    update(
        doi="10.1038/s41467-021-27454-7",
        updates={"lat": 42.94, "lon": 0.14, "particle/m3": 0.233},
    )

    # Caracci2023
    # Recompute number (was rounded) to match mass (measured by study)
    doi = "10.1016/j.jhazmat.2023.131036"
    mask = obs["doi"].fillna("Unknown").str.endswith(doi)
    setting = obs.loc[mask, "setting"].unique()  # sanity check
    if len(setting) > 1:
        raise ValueError(f"Rows to recompute must have same setting; got {setting}")
    setting = setting[0]
    ug_per_particle = PARTICLE_MASS[setting]
    obs.loc[mask, "particle/m3"] = (obs.loc[mask, "ug/m3"] / ug_per_particle).round(15)
    update(
        doi=doi,
        updates={
            "notes": "counts estimated from study-reported mass assuming 100 ng/particle"
        },
    )

    # Ding2021
    # Fix number (was rounded)
    update(doi="10.1016/j.atmosenv.2021.118389", updates={"particle/m3": 0.035})

    # Dris2017
    # Fix lat/lon (was mislocated)
    update(doi="10.1016/j.envpol.2016.12.013", updates={"lat": 48.8, "lon": 2.5})

    # Gaston2020
    # Fix lat/lon (was mislocated); fix number (use count of plastic particles identified
    # via Nile red fluorescence rather than via particle colour)
    update(
        doi="10.1177/0003702820920652",
        updates={
            "lat": 34.17,
            "lon": -118.96,
            "particle/m3": 6.2,
            "notes": "count is particles identified via Nile red fluorescence",
        },
    )

    # Trainic2020
    # Round lat/lon to remove false precision
    doi = "10.1038/s43247-020-00061-y"
    mask = obs["doi"].fillna("Unknown").str.endswith(doi)
    update(
        doi=doi,
        updates={
            "lat": obs.loc[mask, "lat"].round(5),
            "lon": obs.loc[mask, "lon"].round(5),
        },
    )

    # Wang2021
    # Fix number (use mean of all samples)
    update(
        doi="10.1016/j.jhazmat.2021.125477",
        updates={"particle/m3": 0.0039, "notes": "count is mean of all samples"},
    )

    # Drop rows with zero microplastics
    obs = obs.loc[obs["particle/m3"].gt(0)].reset_index(drop=True)

    # Ensure mass = number * average mass per particle
    for setting in ["land", "ocean"]:
        mask = obs["setting"].eq(setting)
        obs.loc[mask, "ug/m3"] = (
            obs.loc[mask, "particle/m3"] * PARTICLE_MASS[setting]
        ).round(15)

    # Subset, reorder columns, and sort
    obs = obs.drop(columns=["type", "year", "mon"])
    # fmt: off
    cols = [
        "study", "doi", "lat", "lon", "particle/m3", "ug/m3", "size_low", "size_high",
        "size_units", "shape", "setting"
    ]
    # fmt: on
    cols = cols + obs.columns.difference(cols, sort=False).to_list()
    obs = obs[cols]
    obs = obs.sort_values(["study", "doi"])

    return obs


def revise_fu2023_depo(obs: pd.DataFrame) -> pd.DataFrame:
    """Fix inconsistencies in original data"""

    CONVERSION = 1e-12 * 1e6 * 365  # t/ug * m2/km2 * days/year

    def update(doi: str, updates: dict) -> None:
        """Update data and possibly recompute mass deposition"""

        mask = obs["doi"].fillna("Unknown").str.endswith(doi)

        if "particle/m2/d" in updates:
            setting = obs.loc[mask, "setting"].unique()
            if len(setting) > 1:
                raise ValueError(f"Rows to update must have same setting; got {setting}")
            setting = setting[0]

            ug_per_particle = PARTICLE_MASS[setting]
            updates["t/km2/yr"] = round(
                updates["particle/m2/d"] * CONVERSION * ug_per_particle, 15
            )

        if "notes" in updates:
            updates["notes"] = (
                obs.loc[mask, "notes"].astype(str) + "; " + updates["notes"]
            )

        for k, v in updates.items():
            obs.loc[mask, k] = v

    # Allen2019
    # Fix lat/lon (was mislocated)
    update(doi="10.1038/s41561-019-0335-5", updates={"lat": 42.804, "lon": 1.419})

    # Brahney2020
    # Fix lat/lon (were mislocated)
    lats = [36.1, 42.9, 43.5, 40.3, 34.1, 40.8, 38.46, 40.1, 39.0, 39.0, 37.6]
    # fmt: off
    lons = [
        -112.2, -109.8, -113.5, -105.6, -116.4, -109.5, -109.8, -105.6, -107.0, -114.2,
        -112.2
    ]
    # fmt: on
    update(doi="10.1126/science.aaz5819", updates={"lat": lats, "lon": lons})

    # Cai2017
    # Fix number (exclude organic particles)
    update(
        doi="10.1007/s11356-017-0116-x",
        updates={
            "particle/m2/d": 36,
            "notes": "count excludes organic particles e.g. cellulose",
        },
    )

    # Dris2016
    # Fix lat/lon (were mislocated)
    update(doi="10.1016/j.marpolbul.2016.01.006", updates={"lat": 48.805, "lon": 2.443})

    # Klein2019
    # Fix lat/lon (mean of all six sample locations); fix number (mean of all samples)
    update(
        doi="10.1016/j.scitotenv.2019.05.405",
        updates={
            "lat": 53.518,
            "lon": 9.923,
            "particle/m2/d": 305.1,
            "notes": "count is mean of all samples",
        },
    )

    # Roblin2020
    # Fix lat/lon (location of bulk deposition sample); fix number (value of bulk
    # deposition sample excluding organic fibers)
    update(
        doi="10.1021/acs.est.0c04000",
        updates={
            "lat": 55.37175,
            "lon": -7.33945,
            "particle/m2/d": 15,
            "notes": (
                "count from bulk deposition sample only"
                + "; count excludes organic fibers e.g. cellulose"
            ),
        },
    )

    # Wright2020
    # Fix lat/lon; fix number (sum of fibers and fragments)
    update(
        doi="10.1016/j.envint.2019.105411",
        updates={"lat": 51.5111, "lon": -0.1171, "particle/m2/d": 771},
    )

    # Unknown studies: adjust number to match mass and round
    # Since we cannot verify the counts, we update them to match the reported masses to
    # minimize the change in the masses when we recompute them from the counts using the
    # assumed mass per particle
    for setting in ["land", "ocean"]:
        mask = obs["study"].str.startswith("Unknown") & obs["setting"].eq(setting)
        obs.loc[mask, "particle/m2/d"] = (
            obs.loc[mask, "t/km2/yr"] / CONVERSION / PARTICLE_MASS[setting]
        ).round(1)

    # Drop duplicate rows (Klein2019, Wright2020)
    obs = obs.drop_duplicates().reset_index(drop=True)

    # Drop rows with zero microplastics
    obs = obs.loc[obs["particle/m2/d"].gt(0)].reset_index(drop=True)

    # Ensure mass = number * unit conversion * average mass per particle
    for setting in ["land", "ocean"]:
        mask = obs["setting"].eq(setting)
        obs.loc[mask, "t/km2/yr"] = (
            obs.loc[mask, "particle/m2/d"] * CONVERSION * PARTICLE_MASS[setting]
        ).round(15)

    # Subset, reorder columns, and sort
    obs = obs.drop(columns=["type", "year", "mon"])
    # fmt: off
    cols = [
        "study", "doi", "lat", "lon", "particle/m2/d", "t/km2/yr", "size_low",
        "size_high", "size_units", "shape", "setting"
    ]
    # fmt: on
    cols = cols + obs.columns.difference(cols, sort=False).to_list()
    obs = obs[cols].sort_values(["study", "doi"])

    return obs


def save_or_load(
    out_name: str,
    fun: Callable,
    out_dir: str = PROCESSED_DIR,
    overwrite: bool = False,
    **kwargs,
) -> pd.DataFrame:
    """Wrapper to skip processing if output file already exists"""

    out_path = os.path.join(out_dir, out_name)
    if os.path.exists(out_path) and not overwrite:
        print(f"File exists: {out_path}")
        return pd.read_csv(out_path)

    result: pd.DataFrame = fun(**kwargs)
    result.to_csv(out_path, index=False)
    print(f"-> {out_path}")

    return result


Introduction
------------

This notebook processes observations of atmospheric microplastic concentration and deposition so that we can use them to constrain our simulations.

The observations were collected from the literature by @Fu2023, who used them to constrain simulations of atmospheric microplastic cycling. We obtained the data from Yanxu Zhang.

Annotate @Fu2023's observations
-------------------------------

@Fu2023's observational dataset does not identify which study each observation was sourced from. We manually match the obserations to studies in the literature (when possible) based on location and microplastic particle count and add metadata taken from the study text:

* Source study
* Observed particle size range
* Observed microplastic shape(s)

### Annotated concentration

In [2]:
conc_annot = save_or_load(
    out_name="obs_fu2023_concentration.csv",
    fun=annotate_fu2023_conc,
    data_dir=DATA_DIR,
    overwrite=OVERWRITE,
)

-> data/processed/obs_fu2023_concentration.csv


In [3]:
# Print summary for reference
clean_styler(
    conc_annot.groupby(["study", "doi", "setting"], dropna=False)
    .agg(
        {
            "lat": "count",
            "size_low": "min",
            "size_high": "max",
            "particle/m3": "mean",
            "ug/m3": "mean",
        }
    )
    .rename(columns={"lat": "n_obs"})
    .style.set_caption("Observed microplastic concentration used by Fu et al. (2023)")
    .format("{:.1f}")
    .format("{:.0f}", subset=["n_obs"])
    .format("{:.3f}", subset=["particle/m3"])
    .format("{:.4f}", subset=["ug/m3"])
)

,,,n_obs,size_low,size_high,particle/m3,ug/m3
study,doi,setting,,,,,
Abbasi2019,https://doi.org/10.1016/j.envpol.2018.10.039,land,1,10.0,500.0,0.700,0.0402
Allen2020,https://doi.org/10.1371/journal.pone.0232746,land,1,5.0,140.0,2.900,0.1665
Allen2021,https://doi.org/10.1038/s41467-021-27454-7,land,1,3.5,53.0,0.230,0.0132
Caracci2023,https://doi.org/10.1016/j.jhazmat.2023.131036,ocean,20,1.0,50.0,0.047,0.0047
Ding2021,https://doi.org/10.1016/j.atmosenv.2021.118389,ocean,1,50.0,2210.0,0.040,0.0035
Dris2017,https://doi.org/10.1016/j.envpol.2016.12.013,land,1,50.0,1650.0,0.900,0.0517
Gaston2020,https://doi.org/10.1177/0003702820920652,land,1,25.0,2061.0,17.000,0.9760
Liu2019,https://doi.org/10.1021/acs.est.9b03427,ocean,3,16.0,2087.0,0.280,0.0280
Liu2020,https://doi.org/10.1016/j.jhazmat.2020.123223,land,9,16.0,2986.0,0.119,0.0068


### Annotated deposition

In [4]:
depo_annot = save_or_load(
    out_name="obs_fu2023_deposition.csv",
    fun=annotate_fu2023_depo,
    data_dir=DATA_DIR,
    overwrite=OVERWRITE,
)

-> data/processed/obs_fu2023_deposition.csv


In [5]:
# Print summary for reference(
clean_styler(
    depo_annot.groupby(["study", "doi", "setting"], dropna=False)
    .agg(
        {
            "lat": "count",
            "size_low": "min",
            "size_high": "max",
            "particle/m2/d": "mean",
            "t/km2/yr": "mean",
        }
    )
    .rename(columns={"lat": "n_obs"})
    .style.set_caption("Observed microplastic deposition used by Fu et al. (2023)")
    .format("{:.1f}")
    .format("{:.0f}", subset=["n_obs"])
    .format("{:.5f}", subset=["t/km2/yr"])
)

,,,n_obs,size_low,size_high,particle/m2/d,t/km2/yr
study,doi,setting,,,,,
Allen2019,https://doi.org/10.1038/s41561-019-0335-5,land,1,10.0,3000.0,365.0,0.00765
Brahney2020,https://doi.org/10.1126/science.aaz5819,land,11,4.0,3000.0,131.9,0.00275
Cai2017,https://doi.org/10.1007/s11356-017-0116-x,land,1,20.0,5000.0,244.0,0.00512
Dris2016,https://doi.org/10.1016/j.marpolbul.2016.01.006,land,1,50.0,3200.0,110.0,0.00231
Klein2019,https://doi.org/10.1016/j.scitotenv.2019.05.405,land,3,20.0,5000.0,324.7,0.00681
Liu2020,https://doi.org/10.1016/j.jhazmat.2020.123223,land,4,16.0,4900.0,50.5,0.00106
Roblin2020,https://doi.org/10.1021/acs.est.0c04000,land,1,50.0,4000.0,80.0,0.00168
Stanton2019,https://doi.org/10.1016/j.scitotenv.2019.02.278,land,1,nan,nan,2.9,0.00006
Tian2020,https://doi.org/10.13671/j.hjkxxb.2020.0067,land,3,nan,nan,172.3,0.00361


Revise @Fu2023's observations
-----------------------------

In some cases the data of @Fu2023 are inconsistent with the study from which they are taken. We update the data to ensure that:

* Lat/lon matches study-reported location
* Number concentration/deposition matches study-reported value

We also recompute mass from number using @Fu2023's assumed average particle mass of 57 ng/particle over land and 100 ng/particle over oceans. We do this because the number and mass values do not always correspond, likely due to rounding. For example, if the data show a number concentration over land of 1 particle/m^3^ and mass concentration of 0.1 µg/m^3^ we change the mass concentration to 0.057.

For the concentration observations sourced from @Caracci2023, which reported microplastic particle mass rather than particle counts, we recompute number from mass assuming 100 ng/particle (all samples were collected over the Atlantic).

### Revised concentration

In [6]:
conc_revised = save_or_load(
    out_name="obs_revised_concentration.csv",
    fun=revise_fu2023_conc,
    obs=conc_annot,
    overwrite=OVERWRITE,
)

-> data/processed/obs_revised_concentration.csv


In [7]:
# Print summary for reference(
clean_styler(
    conc_revised.groupby(["study", "doi", "setting"], dropna=False)
    .agg(
        {
            "lat": "count",
            "size_low": "min",
            "size_high": "max",
            "particle/m3": "mean",
            "ug/m3": "mean",
        }
    )
    .rename(columns={"lat": "n_obs"})
    .style.set_caption(
        "Revised observations of microplastic concentration based on Fu et al. (2023)"
    )
    .format("{:.1f}")
    .format("{:.0f}", subset=["n_obs"])
    .format("{:.3f}", subset=["particle/m3"])
    .format("{:.4f}", subset=["ug/m3"])
)

,,,n_obs,size_low,size_high,particle/m3,ug/m3
study,doi,setting,,,,,
Abbasi2019,https://doi.org/10.1016/j.envpol.2018.10.039,land,1,10.0,500.0,0.657,0.0374
Allen2020,https://doi.org/10.1371/journal.pone.0232746,land,1,5.0,140.0,2.900,0.1653
Allen2021,https://doi.org/10.1038/s41467-021-27454-7,land,1,3.5,53.0,0.233,0.0133
Caracci2023,https://doi.org/10.1016/j.jhazmat.2023.131036,ocean,10,1.0,50.0,0.094,0.0094
Ding2021,https://doi.org/10.1016/j.atmosenv.2021.118389,ocean,1,50.0,2210.0,0.035,0.0035
Dris2017,https://doi.org/10.1016/j.envpol.2016.12.013,land,1,50.0,1650.0,0.900,0.0513
Gaston2020,https://doi.org/10.1177/0003702820920652,land,1,25.0,2061.0,6.200,0.3534
Liu2019,https://doi.org/10.1021/acs.est.9b03427,ocean,3,16.0,2087.0,0.280,0.0280
Liu2020,https://doi.org/10.1016/j.jhazmat.2020.123223,land,9,16.0,2986.0,0.119,0.0068


### Revised deposition

In [8]:
depo_revised = save_or_load(
    out_name="obs_revised_deposition.csv",
    fun=revise_fu2023_depo,
    obs=depo_annot,
    overwrite=OVERWRITE,
)

-> data/processed/obs_revised_deposition.csv


In [9]:
# Print summary for reference(
clean_styler(
    depo_revised.groupby(["study", "doi", "setting"], dropna=False)
    .agg(
        {
            "lat": "count",
            "size_low": "min",
            "size_high": "max",
            "particle/m2/d": "mean",
            "t/km2/yr": "mean",
        }
    )
    .rename(columns={"lat": "n_obs"})
    .style.set_caption(
        "Revised observations of microplastic deposition based on Fu et al. (2023)"
    )
    .format("{:.1f}")
    .format("{:.0f}", subset=["n_obs"])
    .format("{:.5f}", subset=["t/km2/yr"])
)

,,,n_obs,size_low,size_high,particle/m2/d,t/km2/yr
study,doi,setting,,,,,
Allen2019,https://doi.org/10.1038/s41561-019-0335-5,land,1,10.0,3000.0,365.0,0.00759
Brahney2020,https://doi.org/10.1126/science.aaz5819,land,11,4.0,3000.0,131.9,0.00274
Cai2017,https://doi.org/10.1007/s11356-017-0116-x,land,1,20.0,5000.0,36.0,0.00075
Dris2016,https://doi.org/10.1016/j.marpolbul.2016.01.006,land,1,50.0,3200.0,110.0,0.00229
Klein2019,https://doi.org/10.1016/j.scitotenv.2019.05.405,land,1,20.0,5000.0,305.1,0.00635
Liu2020,https://doi.org/10.1016/j.jhazmat.2020.123223,land,4,16.0,4900.0,50.5,0.00105
Roblin2020,https://doi.org/10.1021/acs.est.0c04000,land,1,50.0,4000.0,15.0,0.00031
Stanton2019,https://doi.org/10.1016/j.scitotenv.2019.02.278,land,1,nan,nan,2.9,0.00006
Tian2020,https://doi.org/10.13671/j.hjkxxb.2020.0067,land,3,nan,nan,172.3,0.00359
